# Pandas Notebook 02: Selection, filtering, and indexing

In this notebook, we will practice some of the most useful pandas skills: selecting columns, filtering rows, working with `loc` and `iloc`, and managing indexes.

## Notebook objectives

By the end of this notebook, you should be able to:

- select one or more columns from a DataFrame
- filter rows with one condition or multiple conditions
- use `&`, `|`, and `~` correctly in boolean filtering
- understand why parentheses matter in filters
- explain the difference between `loc` and `iloc`
- select rows and columns together
- set and reset an index
- rename columns clearly
- avoid common beginner mistakes

In [16]:
import pandas as pd

## A Titanic-like dataset for practice

To keep this notebook self-contained, we will create a small Titanic-style dataset directly in the notebook.

Later, you can reuse the same code with a real Kaggle Titanic CSV, for example:

```python
csv_path = "train.csv"
titanic = pd.read_csv(csv_path)
```

In [18]:
# Later, you could replace this sample DataFrame with pd.read_csv("train.csv").
titanic = pd.DataFrame(
    {
        "PassengerId": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "Survived": [0, 1, 1, 1, 0, 0, 0, 0, 1, 1],
        "Pclass": [3, 1, 3, 1, 3, 3, 1, 3, 3, 2],
        "Name": [
            "Braund, Mr. Owen Harris",
            "Cumings, Mrs. John Bradley",
            "Heikkinen, Miss. Laina",
            "Futrelle, Mrs. Jacques Heath",
            "Allen, Mr. William Henry",
            "Moran, Mr. James",
            "McCarthy, Mr. Timothy J",
            "Palsson, Master. Gosta",
            "Johnson, Mrs. Oscar W",
            "Nasser, Miss. Maria",
        ],
        "Sex": ["male", "female", "female", "female", "male", "male", "male", "male", "female", "female"],
        "Age": [22.0, 38.0, 26.0, 35.0, 35.0, None, 54.0, 2.0, 27.0, 14.0],
        "Fare": [7.25, 71.2833, 7.925, 53.1, 8.05, 8.4583, 51.8625, 21.075, 11.1333, 30.0708],
        "Embarked": ["S", "C", "S", "S", "S", "Q", "S", "S", "S", "C"],
    }
)

titanic

,PassengerId,Survived,Pclass,Name,Sex,Age,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley",female,38.0,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath",female,35.0,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,8.0500,S
5,6,0,3,"Moran, Mr. James",male,NaN,8.4583,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,51.8625,S
7,8,0,3,"Palsson, Master. Gosta",male,2.0,21.0750,S
8,9,1,3,"Johnson, Mrs. Oscar W",female,27.0,11.1333,S
9,10,1,2,"Nasser, Miss. Maria",female,14.0,30.0708,C


## 1. Selecting columns

Selecting columns is one of the first things you do when exploring a dataset.

- Use `df["column_name"]` to select one column
- Use `df[["col1", "col2"]]` to select multiple columns

Remember:

- one column returns a `Series`
- multiple columns return a `DataFrame`

In [21]:
print("One column (Series):\n")
print(titanic["Name"])
print("\nMultiple columns (DataFrame):\n")
titanic[["Name", "Sex", "Age"]]

One column (Series):

0         Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley
2          Heikkinen, Miss. Laina
3    Futrelle, Mrs. Jacques Heath
4        Allen, Mr. William Henry
5                Moran, Mr. James
6         McCarthy, Mr. Timothy J
7          Palsson, Master. Gosta
8           Johnson, Mrs. Oscar W
9             Nasser, Miss. Maria
Name: Name, dtype: str

Multiple columns (DataFrame):



,Name,Sex,Age
0,"Braund, Mr. Owen Harris",male,22.0
1,"Cumings, Mrs. John Bradley",female,38.0
2,"Heikkinen, Miss. Laina",female,26.0
3,"Futrelle, Mrs. Jacques Heath",female,35.0
4,"Allen, Mr. William Henry",male,35.0
5,"Moran, Mr. James",male,NaN
6,"McCarthy, Mr. Timothy J",male,54.0
7,"Palsson, Master. Gosta",male,2.0
8,"Johnson, Mrs. Oscar W",female,27.0
9,"Nasser, Miss. Maria",female,14.0


## 2. Filtering rows with conditions

Filtering means keeping only the rows that match a condition.

For example, we may want:

- only passengers who survived
- only female passengers
- only passengers older than 30

In [24]:
survivors = titanic[titanic["Survived"] == 1]
female_passengers = titanic[titanic["Sex"] == "female"]
older_than_30 = titanic[titanic["Age"] > 30]

print("Passengers who survived:\n")
print(survivors[["PassengerId", "Name", "Survived"]])
print("\nFemale passengers:\n")
print(female_passengers[["PassengerId", "Name", "Sex"]])
print("\nPassengers older than 30:\n")
older_than_30[["PassengerId", "Name", "Age"]]

Passengers who survived:

   PassengerId                          Name  Survived
1            2    Cumings, Mrs. John Bradley         1
2            3        Heikkinen, Miss. Laina         1
3            4  Futrelle, Mrs. Jacques Heath         1
8            9         Johnson, Mrs. Oscar W         1
9           10           Nasser, Miss. Maria         1

Female passengers:

   PassengerId                          Name     Sex
1            2    Cumings, Mrs. John Bradley  female
2            3        Heikkinen, Miss. Laina  female
3            4  Futrelle, Mrs. Jacques Heath  female
8            9         Johnson, Mrs. Oscar W  female
9           10           Nasser, Miss. Maria  female

Passengers older than 30:



,PassengerId,Name,Age
1,2,"Cumings, Mrs. John Bradley",38.0
3,4,"Futrelle, Mrs. Jacques Heath",35.0
4,5,"Allen, Mr. William Henry",35.0
6,7,"McCarthy, Mr. Timothy J",54.0


## 3. Multiple conditions with `&`, `|`, and `~`

You can combine conditions to ask more precise questions.

- `&` means **and**
- `|` means **or**
- `~` means **not**

These operators are used often in pandas filtering.

In [34]:
~(titanic["Embarked"] == "S")

0    False
1     True
2    False
3    False
4    False
5     True
6    False
7    False
8    False
9     True
Name: Embarked, dtype: bool

In [38]:
female_survivors = titanic[(titanic["Sex"] == "female") & (titanic["Survived"] == 1)]
first_or_second_class = titanic[(titanic["Pclass"] == 1) | (titanic["Pclass"] == 2)]
not_embarked_s = titanic[~(titanic["Embarked"] == "S")]

print("Female survivors:\n")
print(female_survivors[["Name", "Sex", "Survived"]])
print("\nPassengers in first or second class:\n")
print(first_or_second_class[["PassengerId", "Pclass", "Name"]])
print("\nPassengers who did NOT embark from S:\n")
not_embarked_s[["PassengerId", "Name", "Embarked"]]

Female survivors:

                           Name     Sex  Survived
1    Cumings, Mrs. John Bradley  female         1
2        Heikkinen, Miss. Laina  female         1
3  Futrelle, Mrs. Jacques Heath  female         1
8         Johnson, Mrs. Oscar W  female         1
9           Nasser, Miss. Maria  female         1

Passengers in first or second class:

   PassengerId  Pclass                          Name
1            2       1    Cumings, Mrs. John Bradley
3            4       1  Futrelle, Mrs. Jacques Heath
6            7       1       McCarthy, Mr. Timothy J
9           10       2           Nasser, Miss. Maria

Passengers who did NOT embark from S:



,PassengerId,Name,Embarked
1,2,"Cumings, Mrs. John Bradley",C
5,6,"Moran, Mr. James",Q
9,10,"Nasser, Miss. Maria",C


## 4. Why parentheses matter in boolean filtering

This is one of the most common beginner mistakes.

In pandas, each comparison should usually be wrapped in parentheses before combining them with `&` or `|`.

Correct pattern:

```python
titanic[(titanic["Age"] > 30) & (titanic["Fare"] > 20)]
```

Incorrect pattern:

```python
titanic[titanic["Age"] > 30 & titanic["Fare"] > 20]
```

In [41]:
bad_filter = 'titanic[titanic["Age"] > 30 & titanic["Fare"] > 20]'

print("Incorrect filter example:")
print(bad_filter)
print("\nWhat happens if we try it?")
try:
    eval(bad_filter)
except Exception as error:
    print(type(error).__name__, "-", error)

print("\nCorrect version:\n")
titanic[(titanic["Age"] > 30) & (titanic["Fare"] > 20)][["PassengerId", "Name", "Age", "Fare"]]

Incorrect filter example:
titanic[titanic["Age"] > 30 & titanic["Fare"] > 20]

What happens if we try it?
TypeError - Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]

Correct version:



,PassengerId,Name,Age,Fare
1,2,"Cumings, Mrs. John Bradley",38.0,71.2833
3,4,"Futrelle, Mrs. Jacques Heath",35.0,53.1000
6,7,"McCarthy, Mr. Timothy J",54.0,51.8625


## 5. `loc` vs `iloc`

These are two essential pandas tools for selecting data.

- `loc` uses labels
- `iloc` uses integer positions

They can look similar at first, but they mean different things.

In [61]:
df = titanic.set_index(["PassengerId","Sex"])

In [68]:
print("loc with the default row labels:\n")
print(titanic.loc[0])
print("\nloc with a row slice and selected columns:\n")
print(titanic.loc[0:2, ["Name", "Sex", "Age"]])

titanic_by_id_preview = titanic.set_index(["PassengerId","Sex"])
print("\nloc after using PassengerId as the index:\n")
titanic_by_id_preview.loc[3, ["Name", "Fare"]]

loc with the default row labels:

PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
Fare                              7.25
Embarked                             S
Name: 0, dtype: object

loc with a row slice and selected columns:

                         Name     Sex   Age
0     Braund, Mr. Owen Harris    male  22.0
1  Cumings, Mrs. John Bradley  female  38.0
2      Heikkinen, Miss. Laina  female  26.0

loc after using PassengerId as the index:



,Name,Fare
Sex,,
female,"Heikkinen, Miss. Laina",7.925


In [69]:
print("iloc with one row by position:\n")
print(titanic.iloc[0])
print("\niloc with row positions 0 to 2 and column positions 3 to 5:\n")
print(titanic.iloc[0:3, 3:6])
print("\niloc with a custom set of row and column positions:\n")
titanic.iloc[[1, 4, 7], [3, 5, 6]]

iloc with one row by position:

PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
Fare                              7.25
Embarked                             S
Name: 0, dtype: object

iloc with row positions 0 to 2 and column positions 3 to 5:

                         Name     Sex   Age
0     Braund, Mr. Owen Harris    male  22.0
1  Cumings, Mrs. John Bradley  female  38.0
2      Heikkinen, Miss. Laina  female  26.0

iloc with a custom set of row and column positions:



,Name,Age,Fare
1,"Cumings, Mrs. John Bradley",38.0,71.2833
4,"Allen, Mr. William Henry",35.0,8.0500
7,"Palsson, Master. Gosta",2.0,21.0750


## 6. Selecting rows and columns together

Often, you do not want the full dataset. You want only:

- specific rows that match a condition
- specific columns that matter for the question

This is a very practical pattern in real analysis.

In [70]:
print("Female passengers with only a few useful columns:\n")
print(titanic.loc[titanic["Sex"] == "female", ["PassengerId", "Name", "Age", "Fare"]])
print("\nRows 2 to 5 and selected columns by position:\n")
titanic.iloc[2:6, [0, 3, 5, 6]]

Female passengers with only a few useful columns:

   PassengerId                          Name   Age     Fare
1            2    Cumings, Mrs. John Bradley  38.0  71.2833
2            3        Heikkinen, Miss. Laina  26.0   7.9250
3            4  Futrelle, Mrs. Jacques Heath  35.0  53.1000
8            9         Johnson, Mrs. Oscar W  27.0  11.1333
9           10           Nasser, Miss. Maria  14.0  30.0708

Rows 2 to 5 and selected columns by position:



,PassengerId,Name,Age,Fare
2,3,"Heikkinen, Miss. Laina",26.0,7.9250
3,4,"Futrelle, Mrs. Jacques Heath",35.0,53.1000
4,5,"Allen, Mr. William Henry",35.0,8.0500
5,6,"Moran, Mr. James",NaN,8.4583


## 7. Setting an index

Sometimes a column is a better row label than the default numbers. In this dataset, `PassengerId` is a good candidate.

Setting an index can make data easier to look up.

In [10]:
titanic_by_id = titanic.set_index("PassengerId")

print("DataFrame with PassengerId as the index:\n")
print(titanic_by_id.head())
print("\nPassenger with id 4:\n")
titanic_by_id.loc[4, ["Name", "Survived", "Fare"]]

DataFrame with PassengerId as the index:

             Survived  Pclass                          Name     Sex   Age  \
PassengerId                                                                 
1                   0       3       Braund, Mr. Owen Harris    male  22.0   
2                   1       1    Cumings, Mrs. John Bradley  female  38.0   
3                   1       3        Heikkinen, Miss. Laina  female  26.0   
4                   1       1  Futrelle, Mrs. Jacques Heath  female  35.0   
5                   0       3      Allen, Mr. William Henry    male  35.0   

                Fare Embarked  
PassengerId                    
1             7.2500        S  
2            71.2833        C  
3             7.9250        S  
4            53.1000        S  
5             8.0500        S  

Passenger with id 4:



Name        Futrelle, Mrs. Jacques Heath
Survived                               1
Fare                                53.1
Name: 4, dtype: object

## 8. Resetting an index

If you want to go back to the default numbered index, use `reset_index()`.

In [83]:
titanic_reset = titanic_by_id.reset_index()

print("After resetting the index:\n")
titanic_reset.head()

After resetting the index:



,PassengerId,Survived,Pclass,Name,Sex,Age,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley",female,38.0,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath",female,35.0,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,8.0500,S


## 9. Renaming columns

Renaming columns helps make your data clearer and easier to work with, especially if you want names that are shorter or more descriptive.

In [84]:
renamed_titanic = titanic.rename(
    columns={
        "PassengerId": "passenger_id",
        "Survived": "survived",
        "Pclass": "ticket_class",
    }
)

print("Renamed columns:\n")
renamed_titanic.head()

Renamed columns:



,passenger_id,survived,ticket_class,Name,Sex,Age,Fare,Embarked,mi_indice_deseado
30,1,0,3,"Braund, Mr. Owen Harris",male,22.0,7.2500,S,10
31,2,1,1,"Cumings, Mrs. John Bradley",female,38.0,71.2833,C,11
32,3,1,3,"Heikkinen, Miss. Laina",female,26.0,7.9250,S,12
33,4,1,1,"Futrelle, Mrs. Jacques Heath",female,35.0,53.1000,S,13
34,5,0,3,"Allen, Mr. William Henry",male,35.0,8.0500,S,14


## 10. Common beginner mistakes and how to avoid them

Here are some very common mistakes:

- using `and` or `or` instead of `&` or `|`
- forgetting parentheses around each condition
- using `iloc` with column names instead of positions
- misspelling a column name
- forgetting that methods like `set_index()` and `rename()` return a new DataFrame unless you assign the result

In [13]:
print("Mistake 1: using 'and' instead of '&'\n")
try:
    titanic[(titanic["Age"] > 30) and (titanic["Sex"] == "female")]
except Exception as error:
    print(type(error).__name__, "-", error)

print("\nMistake 2: using iloc with column names\n")
try:
    titanic.iloc[:, ["Name", "Fare"]]
except Exception as error:
    print(type(error).__name__, "-", error)

print("\nCorrect patterns:\n")
print(titanic[(titanic["Age"] > 30) & (titanic["Sex"] == "female")][["Name", "Age", "Sex"]])
print("\nUse loc with column names:\n")
titanic.loc[:, ["Name", "Fare"]].head()

Mistake 1: using 'and' instead of '&'

ValueError - The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

Mistake 2: using iloc with column names

IndexError - .iloc requires numeric indexers, got ['Name' 'Fare']

Correct patterns:

                           Name   Age     Sex
1    Cumings, Mrs. John Bradley  38.0  female
3  Futrelle, Mrs. Jacques Heath  35.0  female

Use loc with column names:



,Name,Fare
0,"Braund, Mr. Owen Harris",7.2500
1,"Cumings, Mrs. John Bradley",71.2833
2,"Heikkinen, Miss. Laina",7.9250
3,"Futrelle, Mrs. Jacques Heath",53.1000
4,"Allen, Mr. William Henry",8.0500


## Practice section

Try these short exercises using `titanic`.

1. Select only the `Name` and `Fare` columns.
2. Filter the rows where `Pclass == 1`.
3. Find passengers who are female and younger than 30.
4. Use `loc` to show rows 0 to 3 and only the `Name`, `Age`, and `Embarked` columns.
5. Set `PassengerId` as the index, then select the passenger with id `7`.
6. Rename `Embarked` to `port`.

In [14]:
# Write your practice answers here.
# Example:
# titanic[["Name", "Fare"]]

## Mini challenge

Answer these questions using filtering and indexing:

1. Which passengers are female survivors? Show only `PassengerId`, `Name`, and `Fare`.
2. Which passengers are in third class and older than 20? Show only `Name`, `Age`, and `Pclass`.
3. Set `PassengerId` as the index and find the `Name` and `Embarked` value for passenger `10`.
4. Which passengers did **not** embark from `S`? Show only `Name` and `Embarked`.
5. Bonus: among passengers with `Fare > 30`, who survived?

In [15]:
# Write your mini challenge solution here.

## Key takeaways

- selecting columns is different from filtering rows, and both are basic pandas skills
- use `&`, `|`, and `~` for pandas boolean filtering, not `and`, `or`, and `not`
- put parentheses around each condition when combining filters
- `loc` works with labels, while `iloc` works with integer positions
- setting an index can make row lookups more meaningful
- resetting an index and renaming columns help keep data clean and readable